# 10 — Descarga de corridas → Drive

Resuelve los 19 accessions de `data/organismos.tsv` a corridas contra la ENA y
baja los `.sra` a `tesis/80_sra/`.

**Las sesiones de Colab se mueren, y eso es lo normal, no la excepción.** Este
notebook está hecho para eso: el estado del trabajo es qué archivos existen en
Drive, así que re-ejecutarlo retoma donde quedó. Usá `LIMITE` para que cada
sesión haga una tanda y termine.

Corré antes `00_setup.ipynb`.

## Preámbulo: montar Drive y clonar el repo

El repo es público, así que el clon no necesita credenciales. **Los notebooks
llaman a los scripts del repo en vez de reimplementarlos**: el criterio de
selección de corridas y el de verificación de ensamblados tienen que vivir en
un solo lugar, o dejan de ser reproducibles.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/tesis')
CLON  = pathlib.Path('/content/tesis')
assert DRIVE.exists(), f'no veo {DRIVE} — ¿montaste la cuenta correcta?'
print('Drive OK:', DRIVE)

In [ ]:
import subprocess

REPO = 'youkonskernel-afk/tesis'
URL_ANON = 'https://github.com/' + REPO + '.git'

_AYUDA = (
    "No pude clonar de forma anonima y no hay GITHUB_TOKEN en los Secrets.",
    "Dos salidas, cualquiera sirve:",
    "  a) hacer el repo publico: Settings -> General -> Change visibility",
    "  b) crear un PAT de solo lectura y guardarlo como GITHUB_TOKEN en el",
    "     panel de Secrets de Colab (la llave a la izquierda), habilitando",
    "     el acceso para este notebook.",
)


def _sin_token(txt, secreto):
    # git incluye la URL en sus mensajes de error, y esa URL lleva el token.
    return txt.replace(secreto, '***') if secreto else txt


def clonar():
    if CLON.exists():
        r = subprocess.run(['git', '-C', str(CLON), 'pull', '--ff-only'],
                           capture_output=True, text=True)
        return 'ya estaba clonado; ' + (r.stdout.strip() or r.stderr.strip())

    # 1. Anonimo. Alcanza si el repo es publico.
    r = subprocess.run(['git', 'clone', '--depth', '1', URL_ANON, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode == 0:
        return 'clon anonimo (el repo es publico)'

    # 2. Con token de los Secrets de Colab. Para repo privado.
    tok = None
    try:
        from google.colab import userdata
        tok = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    if not tok:
        raise RuntimeError(chr(10).join(_AYUDA))

    url = 'https://x-access-token:' + tok + '@github.com/' + REPO + '.git'
    r = subprocess.run(['git', 'clone', '--depth', '1', url, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('el clon con token fallo: ' + _sin_token(r.stderr, tok))

    # Sin esto el token queda escrito en .git/config dentro de la VM.
    subprocess.run(['git', '-C', str(CLON), 'remote', 'set-url', 'origin', URL_ANON],
                   capture_output=True, text=True)
    return 'clon con token (el repo es privado)'


print(clonar())
print(subprocess.run(['git', '-C', str(CLON), 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
import os, glob, shutil
SRA_VER = '3.1.1'
c = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')
if c: os.environ['PATH'] = c[0] + ':' + os.environ['PATH']
for b in ['prefetch', 'vdb-validate', 'jq', 'curl']:
    assert shutil.which(b), f'falta {b} — corré 00_setup.ipynb'

SRA = DRIVE / '80_sra'; SRA.mkdir(parents=True, exist_ok=True)
STAGING = pathlib.Path('/content/sra_staging'); STAGING.mkdir(exist_ok=True)
MANIFIESTO_DRIVE = DRIVE / '00_manifiestos' / 'srr_manifest.tsv'
print('destino :', SRA)
print('staging :', STAGING, f'({shutil.disk_usage("/content").free/1e9:.0f} GB libres)')

## 1. Manifiesto

Se genera con `scripts/fetch_runs.sh manifest`, que consulta la ENA y filtra a
datos de RNA: `library_source = TRANSCRIPTOMIC` —el filtro duro, porque varios
BioProjects mezclan corridas GENOMIC— y `SINGLE` para RNA-Seq, porque el PAIRED
de un proyecto de RNA-Seq no es sRNA-seq.

Se guarda una copia en Drive: el clon es efímero y el manifiesto define el
trabajo pendiente.

In [ ]:
import subprocess, shutil as _sh
if MANIFIESTO_DRIVE.exists():
    print('ya hay manifiesto en Drive; lo reuso.')
    print('Para regenerarlo, borralo primero.')
    _sh.copy(MANIFIESTO_DRIVE, CLON / 'data' / 'srr_manifest.tsv')
else:
    r = subprocess.run(['./scripts/fetch_runs.sh', 'manifest'], cwd=CLON,
                       capture_output=True, text=True)
    print(r.stdout[-4000:]); print(r.stderr[-4000:])
    MANIFIESTO_DRIVE.parent.mkdir(parents=True, exist_ok=True)
    _sh.copy(CLON / 'data' / 'srr_manifest.tsv', MANIFIESTO_DRIVE)
    print('copia guardada en', MANIFIESTO_DRIVE)

## 2. Bajar una tanda

`prefetch` escribe primero en el disco de la VM, se corre `vdb-validate`, y
**recién ahí** se mueve a Drive. Dos motivos: escribir GB directo al FUSE de
Drive es lento e inestable, y un `.sra` truncado no falla ruidosamente —alinea
de menos—, así que el que no valida se descarta y nunca llega al destino.

`LIMITE` acota la tanda. Empezá chico para medir cuánto rinde tu sesión.

In [ ]:
LIMITE = 5          # corridas por tanda
ORGANISMO = ''      # '' = todos; o 'prupe', 'gadmo', ...

env = dict(os.environ,
           SRA_DEST=str(SRA),
           SRA_STAGING=str(STAGING),
           SRA_LEDGER=str(CLON / 'data' / 'sra_md5.tsv'),
           MANIFEST=str(CLON / 'data' / 'srr_manifest.tsv'))

cmd = ['./scripts/fetch_runs.sh', 'prefetch']
if ORGANISMO: cmd.append(ORGANISMO)
cmd += ['-n', str(LIMITE)]

p = subprocess.Popen(cmd, cwd=CLON, env=env, text=True,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for ln in p.stdout:
    print(ln, end='')
p.wait()

## 3. Guardar el ledger

Los md5 van a git, no solo a Drive. Copiá esta salida a `data/sra_md5.tsv` en el
repo y commiteala.

In [ ]:
led = CLON / 'data' / 'sra_md5.tsv'
print(led.read_text() if led.exists() else '(vacío)')

## 4. Repetir

Volvé a correr la celda 2 hasta que `90_estado.ipynb` no muestre faltantes.
Cada tanda retoma sola.

**Colab Free no está pensado para trabajo desatendido largo**: el uso sostenido
lleva a throttling. Conviene espaciar las tandas en vez de encadenarlas.